## Score: 0.81842

## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None
!pip install lightautoml >> None
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu117 >> None

## 2. Импорт библиотек и настройка

In [ ]:
import pandas as pd
import numpy as np

from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка данных

In [ ]:
transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Подготовка данных

In [ ]:
dataset_generated_with_cats['client_num'] = dataset_generated_with_cats['client_num'].astype(str)
df_target['client_num'] = df_target['client_num'].astype(str)

train_data = pd.merge(df_target, dataset_generated_with_cats, on='client_num', how='inner')
test_data = dataset_generated_with_cats[~dataset_generated_with_cats['client_num'].isin(df_target['client_num'])].copy()

In [ ]:
categorical_columns_train = train_data.select_dtypes(include=['category']).columns

for col in categorical_columns_train:
    train_data[col] = train_data[col].astype(str)

categorical_columns_test = test_data.select_dtypes(include=['category']).columns

for col in categorical_columns_test:
    test_data[col] = test_data[col].astype(str)

In [ ]:
cat_features = train_data.select_dtypes(include=['object']).columns.tolist()
cat_features = [col for col in cat_features if col not in ['target', 'client_num']]

## 5. Настройка задачи обучения

In [ ]:
task = Task('multiclass') 

## 6. Обучение модели LightAutoML

In [ ]:
roles = {
    'target': 'target',     
}

if cat_features:
    roles['category'] = cat_features

time_limit = 60*60*7  # 7 часов

automl = TabularAutoML(
    task=task,
    timeout=time_limit,
    gpu_ids='all',    
    reader_params={
        'device': 'gpu'
    }
)

oof_pred = automl.fit_predict(train_data, roles=roles)
print('Обучение завершено.')

## 7. Предсказание на тестовых данных

In [ ]:
test_pred = automl.predict(test_data)

predicted_class_indices = np.argmax(test_pred.data, axis=1)

unique_classes = np.sort(train_data['target'].unique())
inv_class_mapping = {idx: cls for idx, cls in enumerate(unique_classes)}
predicted_labels = [inv_class_mapping[idx] for idx in predicted_class_indices]

submission = pd.DataFrame({
    'client_num': test_data['client_num'],
    'target': predicted_labels
})

submission.to_csv('submission.csv', index=False)
print('Файл submission.csv создан.')